In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Customize high-resolution display for charts
%config InlineBackend.figure_format = 'retina'

# 1. Find analysis files
csv_files = glob.glob("report_metrics_*.csv")
if not csv_files:
    print("No CSV files found starting with 'report_metrics_'")
else:
    print(f"Found {len(csv_files)} files: {csv_files}")

    # 2. Merge data
    dfs = [pd.read_csv(f) for f in csv_files]
    df_all = pd.concat(dfs, ignore_index=True)

    # 3. Preprocessing
    if df_all['Accuracy'].max() <= 1.0:
        df_all['Accuracy'] = df_all['Accuracy'] * 100

    num_rounds_per_task = df_all['Round'].max()
    df_all['Global_Round'] = df_all['Train_Task'] * num_rounds_per_task + df_all['Round']

    print(f"Loaded {df_all.shape[0]} rows of data. Generating charts...")

    # 4. Plot charts
    sns.set_theme(style="whitegrid")
    eval_tasks = sorted(df_all['Eval_Task'].unique())
    num_eval_tasks = len(eval_tasks)

    fig, axes = plt.subplots(nrows=num_eval_tasks, ncols=1, 
                             figsize=(12, 5 * num_eval_tasks), 
                             sharex=True)

    if num_eval_tasks == 1:
        axes = [axes]

    for ax, eval_id in zip(axes, eval_tasks):
        subset = df_all[df_all['Eval_Task'] == eval_id]
        
        sns.lineplot(
            data=subset,
            x='Global_Round',
            y='Accuracy',
            hue='Method',
            linewidth=2.5,
            ax=ax,
            palette="tab10"
        )
        
        ax.set_title(f"Accuracy on Test Set of Task {eval_id}", fontweight='bold', fontsize=14)
        ax.set_ylabel("Accuracy (%)", fontsize=12)
        ax.set_ylim(-5, 105)

        # Draw separator lines for Tasks
        for t_split in df_all['Train_Task'].unique():
            if t_split > 0:
                split_round = t_split * num_rounds_per_task
                ax.axvline(x=split_round, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
                ax.text(split_round + 1, 5, f'Start Task {t_split}', color='red', fontsize=10, rotation=90)
                
        ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', title="Strategies")

    axes[-1].set_xlabel("Global Rounds", fontsize=12)
    plt.tight_layout()

    # Display the chart directly in the notebook
    plt.show()